# EC2

Access key was created before and it is in the .aws folder The accessu key let you connect to AWS services <br>


## 0 Create Key Pair and store it
keypair let you connect to an instance of ec2

In [1]:
import boto3
import os

session = boto3.Session(profile_name="Super_Dayo")
ec2 = session.client("ec2")

key_name = "Super_dayo_keypair"
key_path = f"credentials/{key_name}.pem"

# Check whether key pair exists in AWS
response = ec2.describe_key_pairs(
    Filters=[
        {
            "Name": "key-name",
            "Values": [key_name]
        }
    ]
)

if response["KeyPairs"]:
    print(f"Key pair '{key_name}' already exists in AWS.")

    if os.path.exists(key_path):
        print(f"Private key already exists locally at: {key_path}")
    else:
        print("WARNING: Private key is not on this computer.")
        print("AWS cannot return the private key again.")

else:
    print(f"Creating key pair '{key_name}'...")

    resp = ec2.create_key_pair(KeyName=key_name)

    # Make sure credentials directory exists
    os.makedirs("credentials", exist_ok=True)

    # Save private key
    with open(key_path, "w") as file:
        file.write(resp["KeyMaterial"])

    # SSH requires restrictive permissions
    os.chmod(key_path, 0o400)

    print(f"Private key saved to: {key_path}")

Key pair 'Super_dayo_keypair' already exists in AWS.
AWS cannot return the private key again.


the ssh file.pem is now moved to the .ssh folder at home/d/.ssh

## 1 Create an Amazon EC2 instance

In [2]:
import boto3

session = boto3.Session(profile_name="Super_Dayo") #does not have to be the same as your AWS IAM username shown in the console.
ec2 = session.client("ec2")

# ----------------------------------------------------
# Configuration
# ----------------------------------------------------
image_id = "ami-0f8a61b66d1accaee"
instance_name = "Super_Dayo_instance_ec2"
instance_type = "t3.large"
key_name = "Super_dayo_keypair"

# ----------------------------------------------------
# Check if instance already exists
# ----------------------------------------------------
response = ec2.describe_instances(
    Filters=[
        {
            "Name": "tag:Name",
            "Values": [instance_name]
        },
        {
            "Name": "instance-state-name",
            "Values": [
                "pending",
                "running",
                "stopping",
                "stopped"
            ]
        }
    ]
)


# Flatten reservations into a list of instances
instances = [
    instance
    for reservation in response["Reservations"]
    for instance in reservation["Instances"]
]


# ----------------------------------------------------
# Use existing instance or create new one
# ----------------------------------------------------
if instances:

    instance = instances[0]

    instance_id = instance["InstanceId"]
    state = instance["State"]["Name"]

    print("Instance already exists.")
    print("Instance ID:", instance_id)
    print("State:", state)

else:

    print("Instance does not exist. Creating it...")

    # Get correct root device from AMI
    image_response = ec2.describe_images(ImageIds=[image_id])
    root_device = image_response["Images"][0]["RootDeviceName"]
    
    # Create instance
    ec2_response = ec2.run_instances(
        ImageId=image_id,
        MinCount=1,
        MaxCount=1,
        InstanceType=instance_type,
        KeyName=key_name,

        BlockDeviceMappings=[
            {
                "DeviceName": root_device,
                "Ebs": {
                    "DeleteOnTermination": True,
                    "VolumeSize": 50,
                    "VolumeType": "gp3",
                    "Encrypted": True
                }
            }
        ],

        TagSpecifications=[
            {
                "ResourceType": "instance",
                "Tags": [
                    {
                        "Key": "Name",
                        "Value": instance_name
                    }
                ]
            }
        ]
    )

    instance_id = ec2_response["Instances"][0]["InstanceId"]

    print("Created new instance:", instance_id)

Instance already exists.
Instance ID: i-0b6ed98d8f6acb7ce
State: running


## 2 Create Security Group and add rules to it (only for inbound)
* Security group is needed to for data to get pass the VPC
* Security groups control inbound and outbound traffic of the EC2 instance network interface.
* every EC2 instance must have at least one Security Group associated with it. If no Security Group has been specified during the EC2 instance launch, the default Security Group of the default VPC is associated with the instance

22    SSH<br>
80    HTTP<br>
443   HTTPS<br>
8000  FastAPI / Uvicorn<br>
8501  Streamlit<br>

In [3]:
import boto3

session = boto3.Session(profile_name="Super_Dayo")
ec2 = session.client("ec2")

security_group_name = "SuperDayo"
description = (
    "Security group for Super Dayo."
    "Super_dayo is the EC2 instance used for portfolio projects."
)

# --------------------------------------------------
# 1. Get the default VPC
# --------------------------------------------------
vpc_response = ec2.describe_vpcs(
    Filters=[
        {
            "Name": "is-default",
            "Values": ["true"]
        }
    ]
)

vpc_id = vpc_response["Vpcs"][0]["VpcId"]

print("VPC:", vpc_id)


# --------------------------------------------------
# 2. Check whether security group already exists
# --------------------------------------------------
response = ec2.describe_security_groups(
    Filters=[
        {
            "Name": "group-name",
            "Values": [security_group_name]
        },
        {
            "Name": "vpc-id",
            "Values": [vpc_id]
        }
    ]
)

if response["SecurityGroups"]:

    security_group_id = response["SecurityGroups"][0]["GroupId"]

    print("Security group already exists.")
    print("Security Group ID:", security_group_id)

else:

    print("Creating security group...")

    response = ec2.create_security_group(
        GroupName=security_group_name,
        Description=description,
        VpcId=vpc_id
    )
    security_group_id = response["GroupId"]
    print("Created Security Group:", security_group_id)

# --------------------------------------------------
# 3. Define desired inbound rules
# --------------------------------------------------
desired_rules = [
    {
        # SSH
        "IpProtocol": "tcp",
        "FromPort": 22,
        "ToPort": 22,
        "IpRanges": [
            {
                "CidrIp": "0.0.0.0/0",
                "Description": "SSH access"
            }
        ]
    },
    {
        # HTTP
        "IpProtocol": "tcp",
        "FromPort": 80,
        "ToPort": 80,
        "IpRanges": [
            {
                "CidrIp": "0.0.0.0/0",
                "Description": "HTTP access"
            }
        ]
    }
]


# --------------------------------------------------
# 4. Get current security group rules
# --------------------------------------------------
sg_response = ec2.describe_security_groups(
    GroupIds=[security_group_id]
)

existing_permissions = sg_response["SecurityGroups"][0]["IpPermissions"]


# --------------------------------------------------
# 5. Add only rules that don't already exist
# --------------------------------------------------
for rule in desired_rules:

    rule_exists = False

    for existing_rule in existing_permissions:

        same_protocol = (
            existing_rule.get("IpProtocol") == rule["IpProtocol"]
        )

        same_from_port = (
            existing_rule.get("FromPort") == rule["FromPort"]
        )

        same_to_port = (
            existing_rule.get("ToPort") == rule["ToPort"]
        )

        existing_cidrs = [
            ip_range["CidrIp"]
            for ip_range in existing_rule.get("IpRanges", [])
        ]

        desired_cidr = rule["IpRanges"][0]["CidrIp"]

        if (
            same_protocol
            and same_from_port
            and same_to_port
            and desired_cidr in existing_cidrs
        ):
            rule_exists = True
            break

    if rule_exists:

        print(
            f"Rule already exists: "
            f"port {rule['FromPort']}"
        )

    else:

        ec2.authorize_security_group_ingress(
            GroupId=security_group_id,
            IpPermissions=[rule]
        )

        print(
            f"Added inbound rule: "
            f"port {rule['FromPort']}"
        )


print("\nSecurity Group ID:", security_group_id)

VPC: vpc-0c32dcb0238769b3b
Security group already exists.
Security Group ID: sg-0ba9554a5e88cbaf2
Rule already exists: port 22
Rule already exists: port 80

Security Group ID: sg-0ba9554a5e88cbaf2


## 3 connect new security group to ec2 instance

In [4]:
# add a new one only
ec2.modify_instance_attribute(InstanceId=instance_id, Groups=[security_group_id])
# add a new one to the old one
#ec2.modify_instance_attribute(InstanceId=instrance_id, Groups=[old_gid, security_group_id])

{'ResponseMetadata': {'RequestId': 'a8ed4309-60d9-4135-8124-5f455713bcdb',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'a8ed4309-60d9-4135-8124-5f455713bcdb',
   'cache-control': 'no-cache, no-store',
   'strict-transport-security': 'max-age=31536000; includeSubDomains',
   'content-type': 'text/xml;charset=UTF-8',
   'content-length': '235',
   'date': 'Fri, 02 Oct 2026 05:19:50 GMT',
   'server': 'AmazonEC2'},
  'RetryAttempts': 0}}

## 4. Start, Stop and Delete Amazon EC2 instances

In [5]:
import time

def wait_for_status(instance_id, target_status):
    while True:
        response = ec2.describe_instances(InstanceIds=instance_id)

        status = response['Reservations'][0]['Instances'][0]['State']['Name']

        if status == target_status:
            print("Instance is in {} state".format(target_status))
            break
        
        time.sleep(10)


In [9]:
# start instance
def start_instances(instance_id):
    print("EC2 Instance Start")
    ec2.start_instances(InstanceIds=instance_id)

    wait_for_status(instance_id, 'running')



In [37]:
#start_instances([instance_id])

EC2 Instance Start
Instance is in running state


In [7]:
instance_id

'i-0b6ed98d8f6acb7ce'

In [11]:
# stop instance
def stop_instances(instance_id):
    print("EC2 Instance Stop")
    ec2.stop_instances(InstanceIds=instance_id)

    wait_for_status(instance_id, 'stopped')


In [36]:
#stop_instances([instance_id])

EC2 Instance Stop
Instance is in stopped state


In [13]:
# terminate instance
def terminate_instances(instance_id):
    print("EC2 Instance Termination")
    ec2.terminate_instances(InstanceIds=instance_id)

    wait_for_status(instance_id, 'terminated')



In [14]:
#terminate_instances([instrance_id])

# S3

## 0 List S3 Buckets and Create an S3 Bucket

In [15]:
s3  = session.client("s3")


In [16]:
response=s3.list_buckets()
response

{'ResponseMetadata': {'RequestId': 'T1XXK6PGTPVF2PD1',
  'HostId': 'oGubI2yfVtO0Vi9UQMRUzCMKDP1iJQ34+4Da9xsJx58Nj04BBLeK24HVTnhLWsUzg2PcJ6hUXp8=',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': 'oGubI2yfVtO0Vi9UQMRUzCMKDP1iJQ34+4Da9xsJx58Nj04BBLeK24HVTnhLWsUzg2PcJ6hUXp8=',
   'x-amz-request-id': 'T1XXK6PGTPVF2PD1',
   'date': 'Fri, 02 Oct 2026 05:21:04 GMT',
   'content-type': 'application/xml',
   'transfer-encoding': 'chunked',
   'server': 'AmazonS3'},
  'RetryAttempts': 0},
 'Buckets': [{'Name': 'mlops-kgptalkiewerrtehry',
   'CreationDate': datetime.datetime(2026, 9, 24, 14, 18, 46, tzinfo=tzutc()),
   'BucketArn': 'arn:aws:s3:::mlops-kgptalkiewerrtehry'},
  {'Name': 'super-dayo-409400548716',
   'CreationDate': datetime.datetime(2026, 9, 30, 1, 28, 44, tzinfo=tzutc()),
   'BucketArn': 'arn:aws:s3:::super-dayo-409400548716'}],
 'Owner': {'ID': 'c625dd21f9f17074150bcb6a5d157d1e8c11040edd8a8be0aabc3e312859bbf1'}}

In [17]:
sts = session.client("sts")
account_id = sts.get_caller_identity()["Account"]

bucket_name = f"super-dayo-{account_id}"

In [18]:
def create_bucket(bucket_name):
    response = s3.list_buckets()
    buckets = [buck['Name'] for buck in response['Buckets']]
    if bucket_name not in buckets:
        s3.create_bucket(Bucket=bucket_name)
        print("Bucket is created")

    else:
        print("Bucket already exists in your account!!! Feel free to use it.")

create_bucket(bucket_name)

Bucket already exists in your account!!! Feel free to use it.


## 2. Upload a Single File to S3

In [19]:
import os
# objectname name of the file at s3. if none it will be the same name at s3
def upload_file(file_path, object_name=None):
    if object_name is None:
        object_name = os.path.basename(file_path)

    s3.upload_file(file_path, bucket_name, object_name)

In [20]:
# if you want to change the name
#upload_file("data/index.html", "somefile.html")
#upload_file("Boto3.ipynb")
#upload_file("Boto3.ipynb")
#upload_file("../project.csv")



## 3. Upload a Folder to S3

In [22]:
import os

def upload_directory(directory_path, s3_prefix):
    for root, dirs, files in os.walk(directory_path):

        for file in files:

            file_path = os.path.join(root, file)

            # Path relative to directory being uploaded
            relpath = os.path.relpath(file_path,directory_path)

            # S3 uses / separators
            s3_key = os.path.join(s3_prefix,relpath).replace("\\", "/")
            print(f"Uploading: {file_path}")
            print(f"       to: s3://{bucket_name}/{s3_key}")

            s3.upload_file(file_path,bucket_name,s3_key)


In [23]:
#upload_directory("../docs", "docs")

## 3. List all objects in a bucket

In [25]:
def list_objects():
    response = s3.list_objects_v2(Bucket=bucket_name)
    for obj in response['Contents']:
        print(obj['Key'])



In [26]:
#list_objects()

## 4 Download S3 File to Local System

In [27]:
def download_file(object_name, file_path):
    if not os.path.exists(os.path.dirname(file_path)):
        os.makedirs(os.path.dirname(file_path))

    s3.download_file(bucket_name, object_name, file_path)

In [72]:
#download_file('project.csv', 'S3data_download/project1.csv')

## 5. Download S3 Folder to Local System

In [29]:
import os

import os

def download_directory(s3_prefix, local_directory):

    paginator = s3.get_paginator("list_objects_v2")

    for page in paginator.paginate(
        Bucket=bucket_name,
        Prefix=s3_prefix
    ):
        if "Contents" not in page:
            print(f"No files found under {s3_prefix}")
            return

        for obj in page["Contents"]:
            s3_key = obj["Key"]
            # skip folder placeholder objects
            if s3_key.endswith("/"):
                continue
            # keep entire S3 path
            local_path = os.path.join( local_directory,s3_key)
            # create folders if needed
            os.makedirs(os.path.dirname(local_path),exist_ok=True)
            print(f"Downloading: {s3_key}")
            print(f"          to: {local_path}")
            s3.download_file(bucket_name,s3_key,local_path)

In [30]:
#download_directory("tests/","S3data_download")

## 6. Delete All Files in an S3 Bucket

In [31]:
def delete_objects():
    response = s3.list_objects_v2(Bucket=bucket_name)

    if 'Contents' in response:

        for obj in response['Contents']:
            s3.delete_object(Bucket=bucket_name, Key=obj['Key'])

In [32]:
#delete_objects()

## 7. Delete the bucket

In [33]:
#s3.delete_bucket(Bucket=bucket_name)

# Using the function 

## upload model to S3

### 0. save class name

In [99]:
class_names = [
    "Bioclastic carbonates / Micritic limestone",   # class2 + class13
    "Breccia carbonates",                           # class3
    "Coarse crystal dolostone",                     # class4
    "Coarse crystal limestone",                     # class5
    "Evaporite",                                    # class6
    "Finely crystalline dolostone",                 # class7
    "Finely crystalline limestone",                 # class8
    "Foam carbonates",                              # class9
    "Cemented fracture",                            # class10
    "Botryoidal dolostone",                         # class11
    "Micritic dolostone",                           # class12
    "Mid crystalline dolostone",                    # class14
    "Mid crystalline limestone",                    # class15
    "Oncolite",                                     # class16
    "Oolite",                                       # class17
    "Siliceous carbonates / Stromatolites",         # class18 + class19
    "Stylolite",                                    # class20
    "Thrombolites / Pore",                          # class21 + class22
]

In [103]:
import json
with open("Streamlit/class_names.json", "w") as f:
    json.dump(class_names, f)

### 1. Save a small model configuration

In [104]:
import json

model_config = {
  "architecture": "convnext_tiny",
  "num_classes": 18,
  "image_size": 224,
  "resize_size": 256,
  "remove_bottom_pixels": 90,
  "confidence_threshold": 0.75,
  "top_k": 3,
  "model_filename": "INFERENCE_MERGED__18_class_convnext_tiny_reproduce_adamw_trial21_epoch11.pth",
  "class_names_file": "class_names.json",
}
with open("Streamlit/model_config.json", "w") as f:
    json.dump(model_config, f, indent=4)

### 2 Upload files to S3

In [121]:
import boto3

s3  = session.client("s3")

bucket_name = "super-dayo-409400548716"
prefix = "models/rock-classifier/convnext_tiny_18class"

s3.upload_file(
    "../models/INFERENCE_MERGED__18_class_convnext_tiny_reproduce_adamw_trial21_epoch11.pth",
    bucket_name,
    f"{prefix}/INFERENCE_MERGED__18_class_convnext_tiny_reproduce_adamw_trial21_epoch11.pth"
)

s3.upload_file("Streamlit/class_names.json",bucket_name,f"{prefix}/class_names.json")

s3.upload_file("Streamlit/model_config.json",bucket_name,f"{prefix}/model_config.json")

print("Model artifacts uploaded successfully.")

Model artifacts uploaded successfully.


In [34]:
#upload_directory("../data/carbonate_1223/PPL-1223/test/", "data/Rock-classification/test/")

# Check Check Your Local Machine's Listening Ports

In [122]:
pip install psutil

Note: you may need to restart the kernel to use updated packages.


In [35]:
import psutil

def get_listening_ports():
    print("Local Listening Ports:")
    # Get all active network connections
    for conn in psutil.net_connections(kind='inet'):
        if conn.status == 'LISTEN':
            port = conn.laddr.port
            pid = conn.pid
            print(f"Port: {port} | Process ID: {pid}")



In [ ]:
#get_listening_ports()